# Chapter 5 — Reasoning & Planning (DevOps Diagnostic Agent)

Companion notebook to `skills/reasoning-planning/`. It walks the chapter's
**"DevOps Diagnostic Agent: Pulling It All Together"** section end to end, wiring
the five Ch5 skills into one incident investigation.

**Scenario (the chapter's production walkthrough).** An alert fires: API response
times for the **checkout** service have spiked from **200 ms to 2.5 s**. The
on-call engineer is asleep. The agent takes the case. The AWS account is
fictional (`123456789012`); every boto3 call goes through `moto` mocks so the
notebook runs with zero real AWS credentials.

**The reasoning layer this chapter adds** (the knowledge graph from Ch3 and the
temporal memory from Ch4 are assumed as ground truth). We exercise each skill in
the order the agent uses it:

| Step | Skill | Chapter role |
|------|-------|--------------|
| 1 | `pipeline-architecture-selector` | choose sequential / loop / tree for the incoming query |
| 2 | `investigation-dag-planner` | decompose hypotheses into parallel phases with early termination |
| 3 | `parallel-reconcile-merge` | test independent hypotheses in isolated branches, reconcile |
| 4 | `investigation-dag-planner` | execute the DAG, terminate early on confirmation |
| 5 | `loop-pipeline-router` | route the evidence-collection refine loop |
| 6 | `constraint-guided-plan-validator` | validate the remediation plan against constraints + capability |

Grounded in real boto3 shapes via `moto` (mocked CloudWatch) at step 3.5, whose
result feeds the DAG's early-termination decision at step 4.

> "The graph plans. Events execute. Results reshape the graph." — Ch5


## 1. Load the five Ch5 skills

Each skill lives in its own directory with a self-contained `lib.py` — and each
module is named `lib`. Inserting five paths onto `sys.path` would collide, so we
load each `lib.py` under a distinct module name with `importlib`. The skills are
pure-stdlib, so this is a plain in-process import. (Step 9 also runs the CLIs via
`subprocess` to prove harness-portability.)

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RP_DIR = PROJECT_ROOT / "skills" / "reasoning-planning"


def load_skill_lib(skill_name):
    """Load a skill's lib.py under a unique module name (avoids `lib` collision)."""
    skill_dir = RP_DIR / skill_name
    mod_name = "ch5_" + skill_name.replace("-", "_")
    spec = importlib.util.spec_from_file_location(mod_name, skill_dir / "lib.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = module
    spec.loader.exec_module(module)
    return module


selector = load_skill_lib("pipeline-architecture-selector")
dagplanner = load_skill_lib("investigation-dag-planner")
merger = load_skill_lib("parallel-reconcile-merge")
looprouter = load_skill_lib("loop-pipeline-router")
validator = load_skill_lib("constraint-guided-plan-validator")

print("Loaded 5 Ch5 skills from:", RP_DIR)
for m in (selector, dagplanner, merger, looprouter, validator):
    print("  -", m.__name__)

## 2. Choose the pipeline architecture

*Skill: `pipeline-architecture-selector` (Ch5 Hybrid Architectures, Examples 5-10/5-11).*

The chapter's first design decision: **which pipeline shape does this query
deserve?** Committing to one shape means simple tasks pay the parallel-coordination
tax and hard tasks get under-served. The answer is a routing decision over two
*independent* axes — task **complexity** and answer **uncertainty** (not a single
"more autonomous" dial):

```
if complexity < SIMPLE and uncertainty < LOW:  sequential   # simple + certain
elif uncertainty > HIGH:                        tree         # explore hypotheses
else:                                           loop         # iterative refinement
```

Then a resource-aware wrapper degrades gracefully (Example 5-11): a `tree` under
the memory floor falls back to sequential, a `loop` under the time floor to a
single best-effort pass. Per the chapter, "graceful degradation is a feature, not
an error case."

The flat lookup **"What is the checkout error rate?"** routes sequential. The
open-ended **"Why did checkout latency spike from 200 ms to 2.5 s?"** scores high
uncertainty and routes to a **tree** of parallel hypothesis tests — which is the
architecture the rest of this notebook builds.

In [ ]:
ALERT_QUERY = ("Why did checkout latency spike from 200ms to 2.5s? "
               "Investigate root cause across services.")
FLAT_QUERY = "What is the checkout service error rate right now?"


def show(decision, label):
    flag = "  [DEGRADED]" if decision.degraded else ""
    print(f"{label}")
    print(f"  complexity={decision.complexity}  uncertainty={decision.uncertainty}")
    print(f"  ideal={decision.architecture}  final={decision.final}{flag}")
    print(f"  reason: {decision.reason}\n")


alert_decision = selector.route_query(ALERT_QUERY)
flat_decision = selector.route_query(FLAT_QUERY)
# Same alert, but the host is memory-starved: the tree cannot fan out safely.
degraded_decision = selector.route_query(ALERT_QUERY, available_memory_mb=128)

show(flat_decision, "flat lookup ->")
show(alert_decision, "latency alert ->")
show(degraded_decision, "latency alert under 128MB ->")

**Verification gate.** The flat query routes sequential; the latency alert
routes to a tree; the same alert on a memory-starved host degrades to a
sequential fallback rather than crashing.

In [ ]:
assert flat_decision.final == "sequential", flat_decision
assert alert_decision.architecture == "tree", alert_decision
assert degraded_decision.architecture == "tree" and degraded_decision.final == "sequential_fallback", degraded_decision
assert degraded_decision.degraded and "->" in degraded_decision.reason
print("PASS: flat->sequential, alert->tree, memory-starved alert->sequential_fallback (graceful).")

## 3. Plan the investigation DAG

*Skill: `investigation-dag-planner` (Ch5 Dynamic DAG construction + "Constructing
the Investigation DAG").*

The alert routed to a **tree**, so the agent now decomposes the incident into
hypotheses and organizes them into a DAG. From the chapter's walkthrough, three
hypotheses emerge — ranked by *structural plausibility* (the dependency graph:
checkout depends on payment, which depends on the database) combined with
*historical frequency* (the memory system's consolidated patterns):

1. **database connection pool exhaustion** (highest — caused similar symptoms 3x in 6 months)
2. **payment service memory pressure** (moderate)
3. **network partition between payment and database** (lowest — only worth testing after ruling out #1)

Hypotheses 1 and 2 read *different data sources*, so they run in one **parallel
phase**. Hypothesis 3 *depends on* ruling out #1. The planner's phase model:
the estimated duration of a parallel phase is the **max** of its concurrent tests,
because they run at the same time. This is exactly the DAG the skill ships in
`hypotheses.json`.

In [ ]:
HYPOTHESES = RP_DIR / "investigation-dag-planner" / "hypotheses.json"
tasks = dagplanner.tasks_from_dicts(json.loads(HYPOTHESES.read_text(encoding="utf-8"))["tasks"])
dag = dagplanner.build_investigation_dag(tasks)

for p in dag.phases:
    kind = "PARALLEL" if len(p.task_ids) > 1 else "single"
    print(f"Phase {p.index} [{kind}]  est={p.estimated_duration_s}s")
    for tid in p.task_ids:
        print(f"    - {tid}: {tasks[tid].description}")
print(f"\nparallelism_factor={dag.parallelism_factor}  "
      f"total_est={dag.total_estimated_duration_s}s  "
      f"critical_path={dag.critical_path_len} phases")

**Verification gate.** The two independent hypotheses (`h1`, `h2`) share the
first parallel phase; the network-partition test (`h3`) lands strictly after `h1`;
the parallel-phase duration equals the *max* of its concurrent tests, not their
sum.

In [ ]:
phase0 = set(dag.phases[0].task_ids)
assert phase0 == {"h1_db_pool_exhaustion", "h2_payment_memory_pressure"}, phase0
assert "h3_network_partition" in dag.phases[1].task_ids, dag.phases[1].task_ids
# phase 0 duration = max(h1=8s, h2=6s) = 8s (concurrent), not 14s (sequential)
assert dag.phases[0].estimated_duration_s == 8.0, dag.phases[0]
assert dag.parallelism_factor >= 1
print(f"PASS: h1+h2 parallel in phase 0 (est {dag.phases[0].estimated_duration_s}s = max, not sum); "
      f"h3 gated behind h1; parallelism_factor={dag.parallelism_factor}.")

## 3.5. Test hypotheses in parallel with error isolation

*Skill: `parallel-reconcile-merge` (Ch5 Tree Pipeline + "The architecture of
controlled parallelism").*

Within the parallel phase, each hypothesis is tested in an **isolated branch**.
The chapter's rule: "One branch's failure shouldn't cascade to others or corrupt
shared state." Each branch is a pure function of a *copy* of the context and writes
its findings (and any red flags) to its own channel; only the **merge node**
combines them. Here the network-partition probe raises a `TimeoutError` — it is
caught and recorded, and the two surviving branches still reconcile.

Two completion modes matter: **`partial_coverage`** (a quorum of branches suffices
— "three successful branches out of four provide sufficient coverage") for the
hypothesis phase, and **`all_or_nothing`** for a confirmation phase that requires
every branch.

In [ ]:
def branch_db_pool(ctx):
    # Confirms the top hypothesis; raises the red flag downstream cares about.
    return {"utilization": 1.0, "waiting_threads": 47, "flags": ["pool_exhausted"]}


def branch_payment_mem(ctx):
    return {"heap_pct": 0.42, "flags": []}


def branch_network_partition(ctx):
    raise TimeoutError("probe to payment->db link timed out")


branches = {
    "db_pool": branch_db_pool,
    "payment_mem": branch_payment_mem,
    "network_partition": branch_network_partition,
}

merge = merger.run_parallel_window(branches, {"incident": "checkout-latency"}, mode="partial_coverage")
print("mode:", merge.mode, "| completed:", merge.completed)
print("succeeded:", merge.succeeded)
print("failed (isolated):", merge.failed)
print("flags (unioned):", merge.flags)
print("rationale:", merge.rationale)

**Verification gate.** The crashing branch is isolated (recorded in `failed`,
never propagated); the two survivors clear the majority quorum so the phase
completes; and the `pool_exhausted` red flag from a surviving branch is surfaced,
not dropped by the failed sibling.

In [ ]:
assert "network_partition" in merge.failed, merge.failed
assert set(merge.succeeded) == {"db_pool", "payment_mem"}, merge.succeeded
assert merge.completed, "2/3 should clear the majority quorum"
assert "pool_exhausted" in merge.flags, merge.flags
# a confirmation phase demanding EVERY branch would not complete with a failure
strict = merger.run_parallel_window(branches, {}, mode="all_or_nothing")
assert not strict.completed, "all_or_nothing must fail when any branch failed"
print("PASS: crash isolated, 2/3 quorum completes, pool_exhausted flag surfaced, all_or_nothing correctly blocks.")

## 3.75. Ground the test in real boto3 (moto-mocked CloudWatch)

The branch above returned a stubbed 100% utilization. In production the agent
issues an actual metrics query. We make that call real (real boto3 shapes) but
mocked with `moto`, so the notebook runs against a fictional account with no
credentials. The chapter's confirmation: "The database shows connection pool at
100% utilization with 47 waiting threads."

The values this cell reads back drive the DAG's early-termination decision in the
next step — the metric *is* the evidence that confirms hypothesis 1.

In [ ]:
import os
import datetime

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "testing")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "testing")

import boto3
from moto import mock_aws

FICTIONAL_ACCOUNT_ID = "123456789012"


@mock_aws
def measure_db_connection_pool():
    cw = boto3.client("cloudwatch")
    now = datetime.datetime.utcnow()
    ns = "DevOps/Checkout"
    dims = [{"Name": "Service", "Value": "checkout-db"}]
    # Seed the metrics a real DB exporter would emit during the incident.
    cw.put_metric_data(Namespace=ns, MetricData=[
        {"MetricName": "ConnectionPoolUtilization", "Dimensions": dims,
         "Timestamp": now, "Value": 1.0, "Unit": "Percent"},
        {"MetricName": "WaitingThreads", "Dimensions": dims,
         "Timestamp": now, "Value": 47, "Unit": "Count"},
    ])

    def stat(metric):
        resp = cw.get_metric_statistics(
            Namespace=ns, MetricName=metric, Dimensions=dims,
            StartTime=now - datetime.timedelta(minutes=5),
            EndTime=now + datetime.timedelta(minutes=5),
            Period=300, Statistics=["Average"],
        )
        pts = resp["Datapoints"]
        return pts[0]["Average"] if pts else None

    return {
        "utilization": stat("ConnectionPoolUtilization"),
        "waiting_threads": stat("WaitingThreads"),
        "account": FICTIONAL_ACCOUNT_ID,
    }


db_pool_metrics = measure_db_connection_pool()
print("boto3 + moto CloudWatch (account " + FICTIONAL_ACCOUNT_ID + "):")
for k, v in db_pool_metrics.items():
    print(f"  {k}: {v}")

**Verification gate.** The mocked CloudWatch call returns shaped datapoints:
pool utilization at 100% and 47 waiting threads — the corroborating evidence for
hypothesis 1.

In [ ]:
assert db_pool_metrics["utilization"] == 1.0, db_pool_metrics
assert int(db_pool_metrics["waiting_threads"]) == 47, db_pool_metrics
print("PASS: CloudWatch (moto) returned pool utilization 100% + 47 waiting threads.")

## 4. Execute the DAG with early termination

*Skill: `investigation-dag-planner` again — `execute_with_early_termination`.*

Execution proceeds phase by phase. Per the chapter: "As soon as a hypothesis is
confirmed with sufficient corroborating evidence, there is no need to continue
testing alternatives." The `test_fn` for hypothesis 1 reads the **moto-derived
metric** from the previous step. Because `h1` confirms in phase 0, the whole
downstream phase (`h3` network partition, plus the confirmation task) is skipped —
"Early termination skips hypothesis 3." 

In [ ]:
POOL_EXHAUSTED = db_pool_metrics["utilization"] is not None and db_pool_metrics["utilization"] >= 1.0


def test_fn(task):
    if task.id == "h1_db_pool_exhaustion":
        return (POOL_EXHAUSTED,
                f"pool utilization {db_pool_metrics['utilization']:.0%}, "
                f"{int(db_pool_metrics['waiting_threads'])} waiting threads")
    if task.id == "h2_payment_memory_pressure":
        return (False, "payment service heap nominal (42%)")
    return (False, "not evaluated (early termination)")


exec_trace = dagplanner.execute_with_early_termination(dag, tasks, test_fn)
print(json.dumps(exec_trace, indent=2))

**Verification gate.** Hypothesis 1 is confirmed in phase 0, and at least one
downstream phase is skipped by early termination.

In [ ]:
assert exec_trace["confirmed"] == "h1_db_pool_exhaustion", exec_trace["confirmed"]
assert exec_trace["phases_skipped"] >= 1, exec_trace
print(f"PASS: confirmed {exec_trace['confirmed']} in phase 0; "
      f"{exec_trace['phases_skipped']} phase(s) skipped by early termination.")

## 5. Route the evidence-collection refine loop

*Skill: `loop-pipeline-router` (Ch5 Loop Pipeline + Error-handling strategies,
Examples 5-6/5-9).*

Not every phase confirms on the first pass. When evidence collection returns
"incomplete" — the payment memory profile has not landed yet — the workflow shifts
to a **loop**: request the missing evidence, re-check, and repeat under an explicit
bound (the chapter's `recursion_limit`). The router distinguishes a **correctable**
gap (refine, up to the retry budget, then fall back) from a **fundamental** defect
(terminate with a partial result rather than looping forever).

In [ ]:
# Correctable gap: the payment memory profile arrives on the second collection pass.
state = {"attempts": 0}


def collect_attempt(retry_count):
    state["attempts"] += 1
    return {"evidence_request": f"payment memory profile (attempt {state['attempts']})"}


def check_gap(candidate):
    if state["attempts"] >= 2:
        return True, None
    return False, looprouter.ValidationError(
        looprouter.SEVERITY_CORRECTABLE, "evidence incomplete: payment memory profile missing")


gap_loop = looprouter.run_loop(collect_attempt, check_gap, max_retries=looprouter.DEFAULT_MAX_RETRIES)
print("correctable evidence gap ->", gap_loop["decision"], f"(after {gap_loop['iterations']} refine)")

# Fundamental defect: a self-contradictory plan cannot be refined away.
fundamental = looprouter.run_loop(
    lambda rc: {"plan": "route the same op through sync AND event-driven"},
    lambda c: (False, looprouter.ValidationError(
        looprouter.SEVERITY_FUNDAMENTAL, "plan contradicts itself")),
    max_retries=looprouter.DEFAULT_MAX_RETRIES,
)
print("fundamental defect      ->", fundamental["decision"], f"(after {fundamental['iterations']} refine)")

**Verification gate.** The correctable gap refines then proceeds; the
fundamental defect terminates immediately with a partial result and never
consumes a refine iteration.

In [ ]:
assert gap_loop["decision"] == looprouter.PROCEED, gap_loop
assert gap_loop["iterations"] >= 1, gap_loop
assert fundamental["decision"] == looprouter.TERMINATE_PARTIAL, fundamental
assert fundamental["iterations"] == 0, fundamental
print("PASS: correctable gap -> proceed after refine; fundamental defect -> terminate_with_partial at iteration 0.")

## 6. Validate the remediation plan against constraints + capability

*Skill: `constraint-guided-plan-validator` (Ch5 Constraint-guided planning +
the capability-model filter).*

The confirmed root cause leads to a remediation plan. Before it executes, the
planner validates it against domain **constraints** (step budget, regulatory
deadline, required/forbidden actions) *and* the agent's **capability model** —
"a hypothesis requiring direct database access gets filtered if the agent only has
read-only monitoring permissions." Capability and forbidden-action violations are
**hard**: a plan the agent cannot legally execute is not "mostly fine."

**Plan A** is the conforming remediation: read metrics, read logs, record the
incident (all read-only). **Plan B** oversteps — it proposes a `modify_db` write
the agent is not authorized for, and it blows the 30-day deadline.

In [ ]:
capability = validator.CapabilityModel(
    allowed_actions={"query_metrics", "read_logs", "record_incident", "rollback_config"},
    max_privilege="read",
)
constraints = validator.extract_constraints({
    "max_steps": 6,
    "deadline_days": 30,
    "required_actions": ["record_incident"],
    "forbidden_actions": ["drop_table"],
})

plan_a = validator.steps_from_dicts([
    {"action": "query_metrics", "privilege": "read"},
    {"action": "read_logs", "privilege": "read"},
    {"action": "record_incident", "privilege": "read"},
])
plan_b = validator.steps_from_dicts([
    {"action": "query_metrics", "privilege": "read"},
    {"action": "modify_db", "privilege": "write", "params": {"eta_days": 45}},
])

res_a = validator.verify(plan_a, constraints, capability)
res_b = validator.verify(plan_b, constraints, capability)

print(f"Plan A (read-only): score={res_a.score} passed={res_a.passed}")
for f in res_a.feedback:
    print("   ", f)
print(f"\nPlan B (unauthorized write): score={res_b.score} passed={res_b.passed}")
for f in res_b.feedback:
    print("   ", f)

**Verification gate.** The conforming read-only plan passes with a perfect
score; the plan proposing an unauthorized write fails on a hard capability
violation regardless of its score.

In [ ]:
assert res_a.passed and res_a.score == 1.0, res_a
assert not res_b.passed, "unauthorized write must not pass"
assert any(v.constraint_kind == "capability" for v in res_b.violations), res_b.violations
print("PASS: Plan A conforms (score 1.0); Plan B blocked on capability (unauthorized write).")

## 7. CLI portability check

The chapter's dual-graph point about coding agents applies to these skills: each
carries a hand-rolled CLI so any harness (not just this notebook) can drive it.
We run one skill's `scenario` subcommand via `subprocess` to prove the CLI seam
works from the shell.

In [ ]:
import subprocess

cli = RP_DIR / "investigation-dag-planner" / "cli.py"
proc = subprocess.run(
    [sys.executable, str(cli), "scenario", "latency-investigation"],
    capture_output=True, text=True,
)
print(proc.stdout)
assert proc.returncode == 0, proc.stderr
print("PASS: investigation-dag-planner CLI scenario exits 0.")

## 8. Full benchmark battery across all five skills

Each skill ships a `benchmark` verification gate. We run all five as subprocesses
and require every one to pass — the notebook's own regression net over the skills
it depends on.

In [ ]:
skills = [
    "pipeline-architecture-selector",
    "investigation-dag-planner",
    "parallel-reconcile-merge",
    "loop-pipeline-router",
    "constraint-guided-plan-validator",
]
results = {}
for name in skills:
    cli = RP_DIR / name / "cli.py"
    p = subprocess.run([sys.executable, str(cli), "benchmark"], capture_output=True, text=True)
    tail = p.stdout.strip().splitlines()[-1] if p.stdout.strip() else "(no output)"
    results[name] = (p.returncode, tail)
    print(f"[{'PASS' if p.returncode == 0 else 'FAIL'}] {name:36} {tail}")

assert all(rc == 0 for rc, _ in results.values()), results
print("\nPASS: all five Ch5 skill benchmark batteries green.")

## 9. DevOps diagnostic verdict

The five skills composed into one investigation, exactly as the chapter's
production walkthrough describes.

### Diagnosed root cause
A database configuration deployment two hours before the alert **reduced max
connections from 100 to 20**. Under checkout load the connection pool saturated
(**100% utilization, 47 waiting threads**, confirmed via the mocked CloudWatch
query), starving `checkout` -> `payment` -> `database` and driving latency from
200 ms to 2.5 s. The deployment timestamp correlates precisely with the start of
the spike — a `CAUSED_BY` edge in temporal memory.

### Remediation plan (structured, capability-validated)
1. **Immediate** — roll back the connection-limit configuration change (`rollback_config`, authorized).
2. **Short-term** — add connection-pool utilization monitoring + alerts.
3. **Long-term** — require configuration-change review for database parameters.

### How the skills composed

| Gate | Result |
|------|--------|
| Pipeline architecture chosen for the alert | PASS — routed to **tree** (high uncertainty); degrades to sequential under memory pressure |
| Investigation DAG built | PASS — `h1`+`h2` parallel phase, `h3` gated behind `h1`, phase duration = max not sum |
| Parallel hypothesis tests isolated + reconciled | PASS — crashing branch isolated, 2/3 quorum, `pool_exhausted` flag surfaced |
| Real boto3 call against mocked AWS returns shaped data | PASS — CloudWatch (moto) returned 100% utilization + 47 waiting threads |
| DAG executed with early termination | PASS — `h1` confirmed in phase 0, downstream phase skipped |
| Evidence-collection refine loop routed | PASS — correctable gap -> proceed; fundamental -> terminate_with_partial |
| Remediation plan validated | PASS — conforming plan scores 1.0; unauthorized write blocked on capability |
| All five skill benchmark batteries | PASS |

**Conclusion.** The chapter's claim holds in code: *the graph plans, execution
runs, results reshape the graph*. The same investigation logic that ran in-process
here scales to dozens of concurrent incidents through the event-driven layer the
chapter describes; the reasoning skills are unchanged, only the execution substrate
shifts. The investigation completed in the walkthrough's three minutes and the
on-call engineer slept through it.